# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labanaprince72-a11y/internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook learns a model for the content-refresh lane. The target is the retrospective `trend_direction == "down"` outcome from the anonymized starter slice. To keep the learned model honest, its features are limited to prior-30-day metrics and static content properties; `trend_direction` and `trend_pct` are never features.

The baseline is the fixed Week-4 action score reconstructed from the same repository data. All comparisons use the same grouped held-out clients and the same precision@K metrics. Results are directional decision support, not causal evidence.


## 1. Method choice and why

This is a binary “which pages should be reviewed first?” question. I start with **Logistic Regression** because its direction is readable and it gives a probability for ranking. I also test a **bounded Random Forest** as a stronger non-linear comparator: 160 trees, maximum depth 10, and minimum leaf size 10. The forest is not automatically better just because it is more complex; I select it only when its held-out precision@K and ROC-AUC justify the extra complexity.

The model uses nine fields that are available before the recent outcome window: prior-30-day impressions, clicks, and sessions; content age; days since last update; search volume; competition; word count; and character count. Missingness is imputed inside each pipeline so the test set does not influence training-time preprocessing.


In [1]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd

from sklearn import __version__ as sklearn_version
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
assert DATA_PATH.exists(), f"Expected {DATA_PATH}; run from the repository root."
df = pd.read_csv(DATA_PATH)
required = {
    "content_id", "client_id", "content_type", "trend_direction",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update", "search_volume",
    "competition", "word_count", "char_count", "impressions_90d",
    "clicks_90d", "ctr", "avg_position"
}
assert not (required - set(df.columns)), f"Missing columns: {sorted(required - set(df.columns))}"

TARGET = "observed_decline_outcome"
df[TARGET] = (df["trend_direction"] == "down").astype(int)

# These are pre-decision fields. No trend label or trend-derived field is included.
FEATURES = [
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update", "search_volume",
    "competition", "word_count", "char_count",
]
FORBIDDEN = {"trend_direction", "trend_pct", TARGET, "is_declining_label"}
assert not FORBIDDEN.intersection(FEATURES)

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df["client_id"].copy()
print(f"Rows: {len(df):,}; decline outcome rate: {y.mean():.1%}")
print(f"scikit-learn={sklearn_version}; pandas={pd.__version__}; numpy={np.__version__}")
print(f"Model features ({len(FEATURES)}): {FEATURES}")


Rows: 30,000; decline outcome rate: 54.2%
scikit-learn=1.9.1; pandas=3.0.5; numpy=2.5.3
Model features (9): ['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'search_volume', 'competition', 'word_count', 'char_count']


## 2. Split design

I use one fixed **75/25 GroupShuffleSplit by `client_id`**, with `random_state=42`. Every client belongs entirely to either train or test, so pages from the same client cannot make the held-out result look better through shared client-specific patterns. This is a grouped generalization check, not a time-series forecast.

The baseline and both models are scored on exactly the same held-out rows. The target is used only to evaluate the ranking after the split.


In [2]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
assert train_clients.isdisjoint(test_clients)
assert len(train_idx) + len(test_idx) == len(df)
print(f"Train rows: {len(train_idx):,}; test rows: {len(test_idx):,}")
print(f"Train clients: {len(train_clients)}; test clients: {len(test_clients)}; overlap: {len(train_clients & test_clients)}")
print(f"Train outcome rate: {y.iloc[train_idx].mean():.1%}; test outcome rate: {y.iloc[test_idx].mean():.1%}")
SPLIT_STRATEGY = "75/25 GroupShuffleSplit by client_id, random_state=42"


Train rows: 22,885; test rows: 7,115
Train clients: 24; test clients: 8; overlap: 0
Train outcome rate: 55.0%; test outcome rate: 51.7%


## 3. Train + compare vs my baseline

The Week-4 baseline is reproduced with its fixed hand-written score: visibility, page-1/2 low-CTR opportunity, and staleness. The score uses no fitted weights. It is ranked on the held-out rows, then compared with model probabilities using precision@10, precision@50, precision@100, and ROC-AUC.

The comparison is retrospective: a high precision means more of the top-ranked rows later appeared in the observed decline outcome. It does not prove that the rule or model caused a decline or that an edit will fix it.


In [3]:
def precision_at_k(y_true, scores, k):
    y_arr = np.asarray(y_true)
    score_arr = np.asarray(scores)
    order = np.argsort(-score_arr, kind="mergesort")[:k]
    return float(y_arr[order].mean())

# Reconstruct the Week-4 baseline from its public rule, with deterministic tie-breaks.
visible = df["impressions_90d"] >= 300
high_visibility = df["impressions_90d"] >= 3_000
position_1_to_20 = df["avg_position"].between(1, 20)
low_ctr = df["ctr"] <= 1.0
stale = df["days_since_last_update"] >= 180
baseline_score = (
    np.select([high_visibility, visible], [2, 1], default=0)
    + np.select([visible & position_1_to_20 & low_ctr, visible & position_1_to_20], [2, 1], default=0)
    + np.select([stale, df["days_since_last_update"] >= 90], [2, 1], default=0)
)

models = {
    "logistic_regression": Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=500, class_weight="balanced", random_state=42)),
    ]),
    "random_forest": Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", RandomForestClassifier(
            n_estimators=160, max_depth=10, min_samples_leaf=10,
            class_weight="balanced_subsample", random_state=42, n_jobs=-1
        )),
    ]),
}

results = []
predictions = {}
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]
for name, model in models.items():
    model.fit(X.iloc[train_idx], y_train)
    probability = model.predict_proba(X.iloc[test_idx])[:, 1]
    predictions[name] = probability
    results.append({
        "method": name,
        "precision_at_10": precision_at_k(y_test, probability, 10),
        "precision_at_50": precision_at_k(y_test, probability, 50),
        "precision_at_100": precision_at_k(y_test, probability, 100),
        "roc_auc": float(roc_auc_score(y_test, probability)),
        "accuracy_at_0.5": float(accuracy_score(y_test, probability >= 0.5)),
    })

base_test_score = baseline_score[test_idx]
results.insert(0, {
    "method": "week4_transparent_baseline",
    "precision_at_10": precision_at_k(y_test, base_test_score, 10),
    "precision_at_50": precision_at_k(y_test, base_test_score, 50),
    "precision_at_100": precision_at_k(y_test, base_test_score, 100),
    "roc_auc": float(roc_auc_score(y_test, base_test_score)),
    "accuracy_at_0.5": np.nan,
})
comparison = pd.DataFrame(results)
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# Select by precision@50, the operational review size used by this lane.
model_rows = comparison[comparison["method"] != "week4_transparent_baseline"]
best_model_name = model_rows.sort_values(["precision_at_50", "roc_auc"], ascending=False).iloc[0]["method"]
best_probability = predictions[best_model_name]
print(f"\nSelected model: {best_model_name} (highest held-out precision@50 among tested models).")
print("Comparison uses the same test rows, same split, and same precision@K metrics for baseline and models.")


                    method  precision_at_10  precision_at_50  precision_at_100  roc_auc  accuracy_at_0.5
week4_transparent_baseline            0.600            0.440             0.430    0.500              NaN
       logistic_regression            0.400            0.620             0.610    0.535            0.532
             random_forest            0.600            0.420             0.490    0.652            0.611

Selected model: logistic_regression (highest held-out precision@50 among tested models).
Comparison uses the same test rows, same split, and same precision@K metrics for baseline and models.


## 4. Errors and interpretation

I inspect permutation importance on the held-out rows rather than treating a model's internal split importance as proof. Then I print false positives and false negatives from the selected model. These are examples of uncertainty to investigate, not automatically wrong data.


In [4]:
selected_model = models[best_model_name]
selected_predictions = (best_probability >= 0.5).astype(int)

perm = permutation_importance(
    selected_model, X.iloc[test_idx], y_test,
    scoring="roc_auc", n_repeats=5, random_state=42, n_jobs=-1
)
importance = (
    pd.DataFrame({"feature": FEATURES, "mean_importance": perm.importances_mean, "std_importance": perm.importances_std})
      .sort_values("mean_importance", ascending=False)
      .reset_index(drop=True)
)
print("Permutation importance on the held-out clients (ROC-AUC drop when shuffled):")
print(importance.head(6).to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("Interpretation: a feature matters here only as a predictive association in this slice; it is not a causal driver.")

error_frame = df.iloc[test_idx][[
    "content_id", "client_id", "content_type", TARGET,
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
]].copy()
error_frame["model_probability"] = best_probability
error_frame["prediction"] = selected_predictions
error_frame["error_type"] = np.select(
    [(error_frame[TARGET] == 0) & (error_frame["prediction"] == 1), (error_frame[TARGET] == 1) & (error_frame["prediction"] == 0)],
    ["false_positive", "false_negative"],
    default="correct"
)
errors = error_frame[error_frame["error_type"] != "correct"].copy()
print(f"\nSelected model threshold errors: {len(errors):,} of {len(error_frame):,} test rows; precision={precision_score(y_test, selected_predictions):.3f}; recall={recall_score(y_test, selected_predictions):.3f}")
print("Error counts by content type:")
print(errors.groupby(["error_type", "content_type"], observed=False).size().rename("n").reset_index().to_string(index=False))
print("\nThree concrete errors to investigate:")
print(errors.sort_values("model_probability", ascending=False).head(3).to_string(index=False))

# Reproducible receipts, keeping the large ranked frame out of git.
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
metrics = {
    "rows": int(len(df)),
    "train_rows": int(len(train_idx)),
    "test_rows": int(len(test_idx)),
    "train_clients": int(len(train_clients)),
    "test_clients": int(len(test_clients)),
    "split_strategy": SPLIT_STRATEGY,
    "target": TARGET,
    "features": FEATURES,
    "forbidden_inputs_excluded": sorted(FORBIDDEN),
    "comparison": [{k: (None if pd.isna(v) else float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) else v) for k,v in row.items()} for row in comparison.to_dict("records")],
    "selected_model": best_model_name,
    "permutation_importance_top": [{k: float(v) if isinstance(v, (float, np.floating)) else v for k,v in row.items()} for row in importance.head(6).to_dict("records")],
    "threshold_precision": float(precision_score(y_test, selected_predictions)),
    "threshold_recall": float(recall_score(y_test, selected_predictions)),
    "error_count": int(len(errors)),
}
with open(output_dir / "ml08_model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nWrote {output_dir / 'ml08_model_metrics.json'}")


Permutation importance on the held-out clients (ROC-AUC drop when shuffled):
             feature  mean_importance  std_importance
    content_age_days           0.0368          0.0054
     clicks_prev_30d           0.0025          0.0011
          word_count           0.0011          0.0017
   sessions_prev_30d           0.0006          0.0006
       search_volume           0.0005          0.0018
impressions_prev_30d           0.0002          0.0008
Interpretation: a feature matters here only as a predictive association in this slice; it is not a causal driver.

Selected model threshold errors: 3,331 of 7,115 test rows; precision=0.539; recall=0.644
Error counts by content type:
    error_type       content_type    n
false_negative    keyword article 1307
false_positive comparison article  298
false_positive    keyword article 1726

Three concrete errors to investigate:
          content_id         client_id    content_type  observed_decline_outcome  impressions_prev_30d  clicks_prev_

## Self-check

- [x] Method choice explains why an interpretable model and a bounded non-linear comparator fit the ranking question.
- [x] Split is grouped by `client_id`, fixed with `random_state=42`, and has zero client overlap.
- [x] Week-4 baseline and both models use the same held-out rows and precision@K metrics.
- [x] The model uses prior-window and static fields only; `trend_direction`, `trend_pct`, and the target are excluded from features.
- [x] Comparison includes precision@10, precision@50, precision@100, ROC-AUC, and the test base rate is printed.
- [x] Permutation importance and three concrete errors are shown.
- [x] Metrics are written to `work/outputs/ml08_model_metrics.json`; no bulk prediction CSV is committed.
- [ ] After local verification, commit the notebook and metrics receipt, then submit the public repository URL on the ML-08 card.
